# Why Rainbow Colour Scales Mislead — Measuring It, Not Just Asserting It

**Companion notebook to:** _Why Rainbow Colour Scales Mislead — a Python Follow-up to Tony Ladson's Post_

**Source:** [Rainbow colour scales in hydrologic maps and charts](https://tonyladson.wordpress.com/2016/05/06/rainbow-colour-scales/) — Tony Ladson, 6 May 2016, drawing on Ed Hawkins' [Scrap rainbow colour scales](https://www.nature.com/articles/519291d) (*Nature*, 2015) and Hawkins' [#ShowYourStripes](https://showyourstripes.info/) warming-stripes visualisations.

The claim "rainbow/jet colourmaps mislead" is usually just asserted. This notebook measures it instead, using `colorspacious` (the same library the actual designers of the viridis colourmap used to justify it) to convert each colourmap to CIE L\*a\*b\* space and look at how perceived lightness changes across the colour scale.

**No real data is used here** — this demonstrates a property of colourmaps themselves, independent of any dataset. Where Ed Hawkins' warming stripes are discussed, they're described and linked to, not reproduced — recreating his actual visualisation needs the real global temperature record, which wasn't available to fetch in this environment; approximating it would defeat the point of a notebook about not misrepresenting data.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from colorspacious import cspace_convert
print('colorspacious loaded OK')

colorspacious loaded OK


## 1. Measuring perceptual lightness across a colourmap

A colourmap that's perceptually well-behaved should have lightness (L\*, the CIE Lab lightness channel) change *monotonically* across the scale — light-to-dark or dark-to-light, consistently. If lightness goes up and down and up again as you move through the colourmap, the eye reads false boundaries at the local light/dark transitions, whether or not the underlying data actually has a boundary there.

In [2]:
def colormap_lightness(cmap_name, n=256):
    cmap = plt.get_cmap(cmap_name, n)
    positions = np.linspace(0, 1, n)
    rgb = cmap(positions)[:, :3]
    lab = cspace_convert(rgb, 'sRGB1', 'CIELab')
    return positions, lab[:, 0]  # L* channel

fig, ax = plt.subplots(figsize=(8, 5))
results = {}
for name in ['jet', 'viridis', 'cividis']:
    pos, L = colormap_lightness(name)
    results[name] = (pos, L)
    ax.plot(pos, L, label=name, lw=2)

ax.set_xlabel('Position in colourmap (0 = low value, 1 = high value)')
ax.set_ylabel('Perceptual lightness (CIE L*)')
ax.set_title('Lightness across the colourmap -- monotonic is good', fontsize=11)
ax.legend()
plt.tight_layout()
plt.savefig('../../images/2026-09_colormap-lightness-curves.png', dpi=150, bbox_inches='tight')
plt.show()

print()
for name, (pos, L) in results.items():
    dL = np.diff(L)
    direction_changes = np.sum(np.diff(np.sign(dL)) != 0)
    print(f'{name:10s} L* range=[{L.min():5.1f}, {L.max():5.1f}]  direction changes={direction_changes}'
          + ('  <- NOT monotonic' if direction_changes > 0 else '  (monotonic)'))


jet        L* range=[ 12.9,  95.9]  direction changes=5  <- NOT monotonic
viridis    L* range=[ 14.9,  90.9]  direction changes=0  (monotonic)
cividis    L* range=[ 13.9,  91.2]  direction changes=0  (monotonic)


<Figure size ... with Axes>

`jet` changes lightness direction 5 times across its range — it gets lighter, then darker, then lighter again, repeatedly, as the underlying value increases smoothly. `viridis` and `cividis` are both perfectly monotonic. This isn't a matter of taste; it's a measurable property of the colourmap, independent of any dataset plotted with it.

## 2. What that actually looks like on a smooth field

The clearest demonstration doesn't need real data — a synthetic field with a single smooth peak and *zero real edges* makes the point on its own.

In [3]:
x = np.linspace(-3, 3, 300)
y = np.linspace(-3, 3, 300)
X, Y = np.meshgrid(x, y)
Z = np.exp(-(X**2 + Y**2) / 4) * 100  # smooth Gaussian, 0-100, no real edges anywhere

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, cmap in zip(axes, ['jet', 'viridis']):
    im = ax.pcolormesh(X, Y, Z, cmap=cmap, shading='auto')
    plt.colorbar(im, ax=ax, label='Field value')
    ax.set_title(cmap, fontsize=11)
    ax.set_aspect('equal')
plt.tight_layout()
plt.savefig('../../images/2026-09_colormap-field-comparison.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size ... with Axes>

The `jet` panel shows what reads as a distinct yellow-green ring around the peak, and the dark red centre looks almost like a separate plateau from the orange band around it. None of that is in the data — `Z` is a single smooth Gaussian with a continuous gradient in every direction, no discontinuity anywhere. The `viridis` panel shows the same field as what it actually is: one smooth hill.

## 3. The other half of the lesson: reduce, don't just recolour

Ed Hawkins' [warming stripes](https://showyourstripes.info/) make a different, complementary point: sometimes the fix isn't a better colourmap, it's asking whether the reader needs axes, gridlines and a legend at all, or whether pure colour — one cell per year, ordered left to right, nothing else on the chart — communicates the trend more directly than a conventional line plot would. That's a genuinely different design decision from "pick a better colourmap," and it's worth reading Hawkins' own explanation and looking at the real thing at the link above rather than a synthetic approximation here.

## References

- Ladson, A.R. (2016). [Rainbow colour scales in hydrologic maps and charts](https://tonyladson.wordpress.com/2016/05/06/rainbow-colour-scales/).
- Hawkins, E. (2015). Scrap rainbow colour scales. *Nature* 519, 291.
- Hawkins, E. [#ShowYourStripes](https://showyourstripes.info/).
- Smith, N. & van der Walt, S. — the `colorspacious` library and the perceptual-uniformity analysis behind matplotlib's `viridis` colourmap.